# Vanilla VAE peak fitting on PROPHESY spectra

This notebook uses the 52,500-spectrum labeled PROPHESY ZIP with the same fully connected vanilla VAE encoder and decoder as the atmospheric notebook. A peak head predicts centers, Gaussian widths, and area fractions. Train a separate model for C 1s, S 2p, or O 1s because their energy grids differ. Run from the `spectraVAE` directory.


In [ ]:
from pathlib import Path
import subprocess
import sys
import matplotlib.pyplot as plt
import numpy as np
from prophesy_peakfit.dataset import load_region
from prophesy_peakfit.vanilla_vae import fit_spectrum

ARCHIVE = Path('datasets/prophesy_peakfit.zip')
REGION = 'C1s'  # Change to 'S2p' or 'O1s' for a separate model.
EPOCHS = 100
data = load_region(ARCHIVE, REGION)
print(REGION, 'spectra:', len(data['noisy_counts']), 'points:', len(data['axis_eV']))
print(dict(zip(*np.unique(data['split'], return_counts=True))))


## Train

Training uses the ZIP directly. All scans and photon energies from one source workbook stay in the same split. The best validation checkpoint and peak predictions for the held-out spectra are saved under `outputs/prophesy_vanilla_vae/<region>/`.


In [ ]:
command = [sys.executable, '-m', 'prophesy_peakfit.vanilla_vae',
           '--archive', str(ARCHIVE), '--region', REGION, '--epochs', str(EPOCHS)]
subprocess.run(command, check=True)


## Inspect one held-out peak fit

The red lines are the generating peak centers; the blue lines are the model's predicted centers. The model's clean spectrum is a denoised reconstruction of the simulated peak signal.


In [ ]:
test_index = np.flatnonzero(data['split'] == 'test')[0]
checkpoint = Path('outputs/prophesy_vanilla_vae') / REGION / 'best.pt'
fit = fit_spectrum(checkpoint, data['noisy_counts'][test_index])
axis = data['axis_eV']
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(axis, data['noisy_counts'][test_index], color='0.7', label='Noisy input')
ax.plot(axis, data['clean_counts'][test_index], color='black', label='True clean signal')
ax.plot(axis, fit['clean_counts'], color='tab:blue', label='VAE reconstruction')
for center in data['peak_center_eV'][test_index]:
    ax.axvline(center, color='tab:red', alpha=0.6)
for center in fit['center_eV']:
    ax.axvline(center, color='tab:blue', linestyle='--', alpha=0.6)
ax.set(xlabel='Binding energy (eV)', ylabel='Counts', title=data['sample_id'][test_index])
ax.legend()
plt.show()
print('Predicted centers (eV):', fit['center_eV'])
print('Predicted Gaussian sigma (eV):', fit['sigma_eV'])
print('Predicted peak areas (counts eV):', fit['area_counts_eV'])
